# Interactive Script: **Super-resolve Data Cube**

**Author:** Baturalp Arisoy<br>
**Contact:** baturalp.arisoy@uni-wuerzburg.de - Call me Batu :)

## Overview
Sentinel-2 delivers four bands at 10 m and six more at 20 m. The Sen2SR models sharpen those grids: either to 2.5 m, or from 20 m to a true 10 m.

> **What super-resolution is and is not.** The model infers detail that the sensor did not record. It is trained on real Sentinel-2 and higher-resolution pairs, so the added detail is plausible, but it is a prediction, not a measurement. Treat a super-resolved cube as an interpretation product: good for delineating and for visual work, not as evidence of a feature you cannot see in the native bands. Nothing here changes the radiometry you should quote in an analysis.

## Contents

1. [The Three Models](#1-the-three-models)
2. [RGBN, 10 m to 2.5 m](#2-rgbn-10-m-to-25-m)
3. [Full Spectral, 10 and 20 m to 2.5 m](#3-full-spectral-10-and-20-m-to-25-m)
4. [20 m to 10 m](#4-20-m-to-10-m)
5. [Output Size, Speed and Precision](#5-output-size-speed-and-precision)

---

In [ ]:
from stac2cube import super_resolve_cube, get_stac_layers, open_cube
import matplotlib.pyplot as plt
import numpy as np

## 1. The Three Models

stac2cube ships three of the Sen2SR models. Each one needs a specific set of bands in the cube; the band **order** is handled for you.

| `model_type` | model | in | out | required bands |
|---|---|---|---|---|
| `"rgbn"` | SEN2SRLite_RGBN | 10 m | 2.5 m | `blue, green, red, nir` |
| `"full_spectral"` | SEN2SRLite | 10 and 20 m | 2.5 m | all ten, see below |
| `"20to10"` | SEN2SRLite_Reference_RSWIR_x2 | 20 m | 10 m | all ten, cube built at 10 m |

The ten bands for the last two: `blue, green, red, nir, nir08, rededge1, rededge2, rededge3, swir16, swir22`. Even if you only care about one 20 m band, all ten must be in the cube, because the model reads them together.

Indices stored in the cube are **recalculated** from the super-resolved bands, not resampled and not predicted. An index therefore survives only if the bands it needs are among the model's output: `"rgbn"` allows the 10 m indices `ndvi`, `ndwi`, `savi` and `evi`, while the other two modes also allow the ones built on 20 m bands (`ndmi`, `nbr`, `mndwi`, `ndbi`, `ndre1`, `ndsi`).

Leaving `model_type=None` picks between `"rgbn"` and `"full_spectral"` from the cube's own band list.

The model weights live in `interactive/model/`. Point `model_dir` at the folder that contains it if you run from somewhere else.

## 2. RGBN, 10 m to 2.5 m

The fast one. Use it when you do not need the 20 m bands.

In [ ]:
get_stac_layers(
    mission="s2",
    polygon="../polygons/test.gpkg",
    resolution=10,
    daterange=["2024-04-01", "2024-04-10"],
    bands=["blue", "green", "red", "nir"],
    indices=["ndvi", "ndwi"],
    max_cc=100,
    output="../results/test_sr_input.nc",
    q=True,
)

In [ ]:
super_resolve_cube(
    input_path="../results/test_sr_input.nc",
    output_path=None,          # None writes <input>_sr.nc next to the input
    model_type="rgbn",
    model_dir="..",            # the folder holding the 'model' directory
)

> A super-resolved cube is 16 times the pixels of its input in the 2.5 m modes. Read the crops you need instead of loading whole cubes into the notebook, or the kernel runs out of memory partway through this notebook.

In [ ]:
def stretch(a, p_low=2, p_high=98):
    lo, hi = np.nanpercentile(a, [p_low, p_high])
    return np.clip((a - lo) / (hi - lo), 0, 1)

with open_cube("../results/test_sr_input.nc") as ds:
    native = ds["Time_Series"]
    date = str(native.time.values[0])[:10]
    print("native:", native.shape, native.attrs["pixel_resolution"], "m")
    # Same patch of ground in both, so the crops differ by the resolution factor.
    n = native.sel(time=date, band=["red", "green", "blue"]).values[:, :120, :120].transpose(1, 2, 0)

with open_cube("../results/test_sr_input_sr.nc") as ds:
    sr = ds["Time_Series"]
    print("super :", sr.shape, sr.attrs["pixel_resolution"], "m", "|", sr.attrs["super_resolution"])
    s = sr.sel(time=date, band=["red", "green", "blue"]).values[:, :480, :480].transpose(1, 2, 0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(stretch(n))
axes[0].set_title("Native, 10 m")
axes[1].imshow(stretch(s))
axes[1].set_title("Super-resolved, 2.5 m")
for ax in axes:
    ax.axis("off")
plt.suptitle(date)
plt.show()

## 3. Full Spectral, 10 and 20 m to 2.5 m

Every band ends up on a 2.5 m grid. Sixteen times as many pixels as the input, so check your disk space and your patience first.

In [ ]:
get_stac_layers(
    mission="s2",
    polygon="../polygons/test_clip.gpkg",     # small area on purpose
    resolution=10,
    daterange=["2024-04-01", "2024-04-06"],
    bands=["blue", "green", "red", "nir", "nir08",
           "rededge1", "rededge2", "rededge3", "swir16", "swir22"],
    indices=["ndvi", "ndmi"],
    max_cc=100,
    output="../results/test_sr_full_input.nc",
    q=True,
)

In [ ]:
super_resolve_cube(
    input_path="../results/test_sr_full_input.nc",
    output_path="../results/test_sr_full.nc",
    model_type="full_spectral",
    model_dir="..",
)

## 4. 20 m to 10 m

Sharpens the six 20 m bands (`rededge1`, `rededge2`, `rededge3`, `nir08`, `swir16`, `swir22`) to real 10 m detail, guided by the four native 10 m bands, which pass through untouched.

The output stays on the cube's 10 m grid, so the pixel size does not change and the file does not grow. Much faster and much smaller than full spectral. The cube must be built at 10 m resolution.

In [ ]:
super_resolve_cube(
    input_path="../results/test_sr_full_input.nc",
    output_path="../results/test_sr_20to10.nc",
    model_type="20to10",
    model_dir="..",
)

In [ ]:
with open_cube("../results/test_sr_full_input.nc") as ds:
    date20 = str(ds["Time_Series"].time.values[0])[:10]
    before20 = ds["Time_Series"].sel(time=date20, band="swir16").values

with open_cube("../results/test_sr_20to10.nc") as ds:
    after20 = ds["Time_Series"].sel(time=date20, band="swir16").values

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(stretch(before20), cmap="gray")
axes[0].set_title("swir16, 20 m resampled to the 10 m grid")
axes[1].imshow(stretch(after20), cmap="gray")
axes[1].set_title("swir16, sharpened to 10 m")
for ax in axes:
    ax.axis("off")
plt.suptitle(date20)
plt.show()

## 5. Output Size, Speed and Precision

| parameter | what it does |
|---|---|
| `pack_to_int16` | On by default. Stores values as int16 with a CF scale and offset, the same scheme Sentinel-2 uses natively: about half the size at almost no time cost, and readers unpack it transparently. If the data range is too wide to hold 1e-4 precision, packing is skipped automatically. Set `False` for bit-exact float32. |
| `compress` | Off by default. zlib on top of the packing: roughly 1.25x smaller for roughly 10x slower writing. Worth it for archiving, not for a working file. |
| `batch_size` | Patches per forward pass. `None` picks a size from free GPU memory and backs off automatically on out-of-memory. |
| `precision` | `"auto"` (= `fp32`) by default. `fp16` and `bf16` run faster on CUDA; any batch that comes back non-finite is redone in fp32, so bright surfaces such as snow and cloud no longer end up as holes. |
| `nan_pixel_buffer` | Extra growth of the no-data and cloud mask, in output pixels. `0` by default, so every pixel you put in comes back out. Raise it if you want a margin around cloud masks anyway, for instance because the mask itself runs a pixel or two tight. |
| `edge_crop_px` | Extra pixels trimmed off each side, `0` by default. Shrinks the cube, so leave it alone unless you specifically want a hard margin. |
| `vrt` | Also write a QGIS band-labelling `.vrt` next to a NetCDF output. |

The output container follows the extension of `output_path`, and with no `output_path` it follows the input's. Writing to `.zarr` works, with one caveat: Zarr attributes are JSON and cannot hold a float32 scale factor, so a packed store decodes to float64. The values match the NetCDF decode within float32 rounding, but not bit for bit.

> **Running several models in one session.** Each call loads its own model onto the GPU, and the memory is not handed back between calls. On a card with 8 GB or less, running all three modes one after another in the same kernel can kill it. Restart the kernel between models, or run one mode per session.

In [ ]:
super_resolve_cube(
    input_path="../results/test_sr_full_input.nc",
    output_path="../results/test_sr_exact.nc",
    model_type="20to10",
    model_dir="..",
    pack_to_int16=False,     # bit-exact float32
    precision="fp32",
    vrt=True,
)

> Super-resolution is not reversible and it is not a preprocessing step to run by default. Keep the native cube. Everything downstream that measures reflectance, cloud percentage or a spectral index should read the native cube; use the super-resolved one for delineation and display.